# Kaggle Data for Recommendation Useacase Portfolio

## Initialization Module

In [ ]:
!pip install -q kaggle cryptography

## Encrypting and Decrypting

In [ ]:
from cryptography.fernet import Fernet
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC as Derived
from cryptography.hazmat.backends import default_backend
from getpass import getpass
import zipfile
import base64
import os

In [ ]:
def ForKeys(password: str, salt: bytes) -> bytes:
    kdf = Derived(algorithm=hashes.SHA256(),
                length=32,
                salt = salt,
                iterations=100000,
                backend=default_backend(),
                )
    key = base64.urlsafe_b64encode(kdf.derive(password.encode()))
    return key

In [ ]:
def EncoderPass(message: str, password: str = None) -> str:
    if password is None:
        password = getpass("Password: ")
    salt = os.urandom(16)
    key = ForKeys(password, salt)
    f = Fernet(key)
    encrypted_data = f.encrypt(message.encode())
    Hasil = base64.urlsafe_b64encode(salt + encrypted_data).decode()
    return Hasil

In [ ]:
def DecoderPass(Enkripsi: str, password: str = None) -> str:
    if password is None:
        password = getpass("Password: ")
    decoded_data = base64.urlsafe_b64decode(Enkripsi.encode())
    salt = decoded_data[:16]
    encrypted_data = decoded_data[16:]
    key = ForKeys(password, salt)
    f = Fernet(key)
    decrypted_data = f.decrypt(encrypted_data)
    Hasil = decrypted_data.decode()
    return Hasil

In [ ]:
username_Encripted = '5uvtqVoPdK6BDPnu5gT_zWdBQUFBQUJwT0F6U1BHMl9OQUYtY2VSOE5KS3lFTFdRWFk3WmFtQk5ORDM2V3FGWUh6TnJNZE1kdnpqNnFzVFMwNDRmbERwSXJSM3RUeFlhakhqZ3JNeHJteXR5SG92TkZRPT0='
APItoken_Encripted = '7rxcrYPUgIOHRBjYj0O3WWdBQUFBQUJwT0F5a3hQSnhabFRJdjlkRktfcVBTVGxwenFQQWVwc1d5dnVuWHlLRUtOblVFMlUyZU1OOVB5UUpPMEhDdFJ4eXdpWUUzbkh0cHpua1UzUEFKWEhLR1ViaFJaZFBWd3BpbngtaUFCb0dIZTJIZnhmSTNUVFNocmZ5SkpHbDJEQ1hRN1Vo'

In [ ]:
username = DecoderPass(username_Encripted)

Password: ··········


In [ ]:
APItoken = DecoderPass(APItoken_Encripted)

Password: ··········


## Init Kaggle Dataset

In [ ]:
import json

auth = {"username": username,"key": APItoken}
with open('kaggle.json', 'w+') as data:
    json.dump(auth, data)

In [ ]:
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 770 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d andrexibiza/grocery-sales-dataset
!ls -lha

Dataset URL: https://www.kaggle.com/datasets/andrexibiza/grocery-sales-dataset
License(s): CC0-1.0
 92% 206M/223M [00:00<00:00, 714MB/s] 
100% 223M/223M [00:00<00:00, 668MB/s]
total 224M
drwxr-xr-x 1 root root 4.0K Dec 22 03:22 .
drwxr-xr-x 1 root root 4.0K Dec 22 03:20 ..
drwxr-xr-x 4 root root 4.0K Dec 11 14:34 .config
-rw-r--r-- 1 root root 224M Jan 31  2025 grocery-sales-dataset.zip
drwxr-xr-x 1 root root 4.0K Dec 11 14:34 sample_data


In [ ]:
def Unzip(Zips       : str,
          password   : str = None,
          DirExtract : str = '.',
         ) -> None:
    if not os.path.exists(Zips):
        print(f'Error: Zip file not found at {Zips}')
        return None

    if password is None:
        password = getpass('Masukkan password: ')
    os.makedirs(DirExtract, exist_ok = True)
    Hasil = False
    try:
        with zipfile.ZipFile(Zips, 'r') as zf:
            if len(str(password)):
                zf.extractall(path=DirExtract, pwd = password.encode())
            else:
                zf.extractall(path=DirExtract)
        print(f'Successfully unzipped {Zips} to {DirExtract}/')
        Hasil = True

    except zipfile.BadZipFile:
        print(f'Error: "{Zips}" is not a valid zip file or is corrupted.')

    except RuntimeError as err:
        print(f'Error unzipping {Zips}: {err}')
        if 'Bad password' in str(err):
            print('Check password anda lagi: ')

    except Exception as arc:
        print(f'An unexpected error occurred: {arc}.')

    finally:
        return Hasil

In [ ]:
Status = Unzip(Zips = 'grocery-sales-dataset.zip',
               DirExtract = 'GrocerySales')
print(Status)

Masukkan password: ··········
Successfully unzipped grocery-sales-dataset.zip to GrocerySales/
True


## Read the Data

In [ ]:
os.chdir('/content/GrocerySales')

In [ ]:
Grosir = os.listdir('.')

print(f"Current working directory: {os.getcwd()}")
print("Files in 'GrocerySales' directory:")
fpat = list()
for file in Grosir:
    print(file)
    x = os.path.realpath(os.path.join(os.getcwd(), file))
    fpat.append(x)

Current working directory: /content/GrocerySales
Files in 'GrocerySales' directory:
categories.csv
cities.csv
countries.csv
sales.csv
employees.csv
products.csv
customers.csv


## Data Countries

In [ ]:
import pandas as pd

Countries = pd.read_csv('countries.csv')
display(Countries.sample(4))

print()

display(Countries.describe())

print()

display(Countries.describe(include = 'object'))

,CountryID,CountryName,CountryCode
184,185,Costa Rica,TM
96,97,Tajikistan,BG
132,133,Jordan,SD
78,79,Egypt,GQ


,CountryID
count,206.000000
mean,103.500000
std,59.611241
min,1.000000
25%,52.250000
50%,103.500000
75%,154.750000
max,206.000000


,CountryName,CountryCode
count,206,205
unique,206,205
top,Armenia,AN
freq,1,1


## Data City

Data `city` berisi informasi mengenai nama kota dan kode pos. Data ini menginformasikan `CityID` dan `CountryID` yang akan dipakai pada bagian logistik atau penjualan barang. Tujuan data ini adalah penyambung dari alamat tujuan dan atau asal barang. Nama-nama kota disini lebih dikenal sebagai nama kota di USA.

In [ ]:
City = pd.read_csv('cities.csv')
display(City.sample(4))

print()

display(City.describe())

print()

display(City.describe(include = 'object'))

,CityID,CityName,Zipcode,CountryID
90,91,Jacksonville,68274,32
87,88,Seattle,20135,32
55,56,Stockton,46777,32
3,4,Fremont,20641,32


,CityID,Zipcode,CountryID
count,96.000000,96.000000,96.0
mean,48.500000,51245.302083,32.0
std,27.856777,29883.397049,0.0
min,1.000000,157.000000,32.0
25%,24.750000,21780.500000,32.0
50%,48.500000,51700.000000,32.0
75%,72.250000,79867.750000,32.0
max,96.000000,97859.000000,32.0


,CityName
count,96
unique,96
top,Dayton
freq,1


## Data Customer

Informasi dari tabel data ini terkait dengan Nama lengkap pelanggan beserta alamatnya. Tabel data ini bisa disambungkan dengan data `City` karena adanya `CityID`. Terdapat `CustomerID` yang dapat disambungkan pada data transaksi. Dan data ini, namanya tidak di-encrypted. Ada kemungkinan, data ini adalah data hasil generated.

In [ ]:
customers = pd.read_csv("customers.csv")
display(customers.sample(4))
print()
display(customers.describe())
print()
display(customers.describe(include = 'object'))

,CustomerID,FirstName,MiddleInitial,LastName,CityID,Address
67205,67206,Roy,R,Davidson,51,53 North Oak Avenue
26556,26557,Laurie,I,Wong,50,28 Oak Way
84899,84900,Dexter,M,Sanchez,81,15 Oak Freeway
90900,90901,Damian,B,Delgado,75,66 First Road


,CustomerID,CityID
count,98759.000000,98759.000000
mean,49380.000000,48.557661
std,28509.411955,27.660082
min,1.000000,1.000000
25%,24690.500000,25.000000
50%,49380.000000,49.000000
75%,74069.500000,72.000000
max,98759.000000,96.000000


,FirstName,MiddleInitial,LastName,Address
count,98759,97782,98759,98759
unique,969,26,995,80134
top,Randi,J,Dorsey,16 Milton Road
freq,141,3826,134,10


## Data Category

Data ini sepertinya berisi informasi subjectif dari pembuat data terkait dengan categori produk yang terjual yang disambungkan pada data transaksi.

In [ ]:
categories = pd.read_csv('categories.csv')
display(categories.sample(4))
print()
display(categories.describe())
print()
display(categories.describe(include = 'object'))

,CategoryID,CategoryName
4,5,Beverages
5,6,Seafood
3,4,Dairy
0,1,Confections


,CategoryID
count,11.000000
mean,6.000000
std,3.316625
min,1.000000
25%,3.500000
50%,6.000000
75%,8.500000
max,11.000000


,CategoryName
count,11
unique,11
top,Confections
freq,1


## Product Data

Informasi yang disediakan dalam data ini terkait dengan barang-barang yang dijual. Key primary pada tabel data ini adalah `ProductID` dengan penyambungnya pada data kategori yakni `CategoryID`. Ada beberapa hal yang perlu diperhatikan. `ProductName`, `Price`, dan `ModifyDate` memiliki makna yang sudah jelas. Namun untuk informasi mengenai `Resistant`, `IsAllergic`, dan `VitalityDays`, masih perlu pengecekan berulang pada _source data_. Tebakan saya adalah:
1. `Resistant` adalah informasi yang mungkin terkait dengan ketahanan lamanya produk itu dapat simpan. Namun beberapa barang sulit dikategorikan bagaimana barang dapat disimpan, tercermin dari kategori resistant sebagai `unknown`.
2. Untuk informasi `IsAllergic`, mungkin dapat diartikan bahwa beberapa produk dapat memberikan reaksi alergi kepada beberapa konsumen tertentu seperti gatal-gatal, pusing, atau lainnya. Data ini adalah data boolean category dengan nilai `True` dan `False`.
3. `VitalityDays` information typically refers to the shelf life or freshness period of a product. It indicates the number of days the product remains effective, usable, or fresh before it is no longer suitable for sale or consumption.

In [ ]:
products = pd.read_csv('products.csv')
display(products.sample(4))
print()
display(products.describe())
print()
display(products.describe(include = 'object'))

,ProductID,ProductName,Price,CategoryID,Class,ModifyDate,Resistant,IsAllergic,VitalityDays
113,114,Veal - Osso Bucco,79.5638,8,High,2017-07-11 15:13:04.140,Weak,True,93.0
336,337,Bread - Rye,83.5373,3,Low,2017-06-21 05:50:53.390,Durable,True,0.0
27,28,Sobe - Tropical Energy,12.3153,9,High,2017-10-25 04:23:48.230,Durable,True,0.0
298,299,Fuji Apples,80.1371,4,Medium,2018-02-24 02:57:20.510,Unknown,False,0.0


,ProductID,Price,CategoryID,VitalityDays
count,452.000000,452.000000,452.000000,452.000000
mean,226.500000,50.801471,5.862832,26.030973
std,130.625419,28.616724,3.271694,39.061200
min,1.000000,0.044900,1.000000,0.000000
25%,113.750000,26.504350,3.000000,0.000000
50%,226.500000,52.499500,6.000000,0.000000
75%,339.250000,75.496450,9.000000,52.500000
max,452.000000,99.875500,11.000000,120.000000


,ProductName,Class,ModifyDate,Resistant,IsAllergic
count,452,452,452,452,452
unique,452,3,452,3,3
top,Napkin White - Starched,Medium,2017-01-06 14:39:42.890,Durable,False
freq,1,156,1,164,165


## Data Employee

bagian data ini menjelaskan terkait dengan para karyawan sales marketing yang bekerja pada perusahaan yang terikait. Data ini dapat disambungkan pada tabel alamat yang sudah disediakan.

In [ ]:
employees = pd.read_csv('employees.csv')
display(employees.sample(4))
print()
display(employees.describe())
print()
display(employees.describe(include = 'object'))

,EmployeeID,FirstName,MiddleInitial,LastName,BirthDate,Gender,CityID,HireDate
0,1,Nicole,T,Fuller,1981-03-07 00:00:00.000,F,80,2011-06-20 07:15:36.920
10,11,Sonya,E,Dickson,1976-01-14 00:00:00.000,F,12,2016-08-10 15:59:30.360
6,7,Chadwick,P,Cook,1970-05-02 00:00:00.000,M,39,2016-07-10 06:22:00.670
15,16,Chadwick,U,Walton,1951-07-07 00:00:00.000,M,28,2017-02-10 11:21:26.650


,EmployeeID,CityID
count,23.00000,23.000000
mean,12.00000,43.782609
std,6.78233,26.381737
min,1.00000,4.000000
25%,6.50000,20.500000
50%,12.00000,39.000000
75%,17.50000,65.000000
max,23.00000,92.000000


,FirstName,MiddleInitial,LastName,BirthDate,Gender,HireDate
count,23,23,23,23,23,23
unique,22,14,23,23,2,23
top,Chadwick,E,Fuller,1981-03-07 00:00:00.000,M,2011-06-20 07:15:36.920
freq,2,3,1,1,15,1


## Data Sales

Untuk data penjualan ini, informasi utama adalah relasi antara salesPerson dan ProdukID dari setiap benda/barang yang sudah di-order.

In [ ]:
sales = pd.read_csv('sales.csv')
display(sales.sample(4))
print()
display(sales.describe())
print()
display(sales.describe(include = 'object'))

,SalesID,SalesPersonID,CustomerID,ProductID,Quantity,Discount,TotalPrice,SalesDate,TransactionNumber
6458686,6458687,3,57093,117,15,0.1,0.0,2018-04-26 18:25:09.510,K5W5JUZRTXWBA130BG0N
6589345,6589346,18,59790,397,16,0.2,0.0,2018-02-15 13:51:04.460,J8QFGNCKQ50PC92BPZGU
4909983,4909984,22,62416,90,16,0.0,0.0,2018-03-10 22:24:02.920,2QH3O7S0CNK9F347YSK9
1100911,1100912,8,87757,318,23,0.0,0.0,2018-05-07 12:03:05.660,RSXKLX9HPQKCEQ1DGV0J


,SalesID,SalesPersonID,CustomerID,ProductID,Quantity,Discount,TotalPrice
count,6.758125e+06,6.758125e+06,6.758125e+06,6.758125e+06,6.758125e+06,6.758125e+06,6758125.0
mean,3.379063e+06,1.199972e+01,4.939567e+04,2.265474e+02,1.300401e+01,2.996787e-02,0.0
std,1.950903e+06,6.632689e+00,2.850504e+04,1.304744e+02,7.209701e+00,6.398096e-02,0.0
min,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,0.0
25%,1.689532e+06,6.000000e+00,2.470700e+04,1.140000e+02,7.000000e+00,0.000000e+00,0.0
50%,3.379063e+06,1.200000e+01,4.941400e+04,2.270000e+02,1.300000e+01,0.000000e+00,0.0
75%,5.068594e+06,1.800000e+01,7.407500e+04,3.400000e+02,1.900000e+01,0.000000e+00,0.0
max,6.758125e+06,2.300000e+01,9.875900e+04,4.520000e+02,2.500000e+01,2.000000e-01,0.0


,SalesDate,TransactionNumber
count,6690599,6758125
unique,6670068,6758125
top,2018-02-22 00:55:15.420,RRBD7S603UIC73B631OF
freq,3,1


# Export to Database

In [ ]:
!pip install -q mermaid-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 21.0 MB/s eta 0:00:00


In [ ]:
from mermaid import Mermaid

MyDiagram = """
graph LR
    subgraph Raw_Data [Main Input Data]
        B[("Sales Data\n6,758,125 row")]
        C[('Product Data\n452 row')]
        D[('Customer Data\n98,759 row')]
    end

    subgraph Location_Hierarchy [Geographic and Suppot Data]
        E[('City Data\n196 row')]
        F[('Country Data\n206 row')]
        I[('Category Data\n11 row\nIf needed')]
    end

    subgraph Staff [HR Data]
        G[('Employee Data\n23 row')]
    end

    %% Join Logic
    B --->|Left Join| C
    B --->|Left Join| D
    C --->|Left Join| E
    D --->|Left Join| E
    E --->|Left Join| F
    I -.->F
    F --->|Left Join| G

    G --> H(('Final Joined Data\nmust be 6,758,125 row'))

    %% Background Styles (Subgraphs)
    style Raw_Data fill:#f2f2f2,stroke:#ccc,stroke-width:1px
    style Location_Hierarchy fill:#f2f2f2,stroke:#ccc,stroke-width:1px
    style Staff fill:#f2f2f2,stroke:#ccc,stroke-width:1px

    %% Cool Box Colors
    style B fill:#00d2ff,stroke:#007bb5,color:#fff
    style C fill:#00d2ff,stroke:#007bb5,color:#fff
    style D fill:#00d2ff,stroke:#007bb5,color:#fff

    style E fill:#a8ff78,stroke:#59ab2d
    style F fill:#a8ff78,stroke:#59ab2d
    style I fill:#a8ff78,stroke:#59ab2d

    style G fill:#ff9a9e,stroke:#d66d75

    style H fill:#c64b02,stroke:#667eea,color:#fff,stroke-width:3px
"""

Mermaid(MyDiagram)

In [ ]:
from duckdb import connect as dcon

dc = dcon('GrocerySales.db')
dc.execute('CREATE SCHEMA IF NOT EXISTS GrocerySales')

In [ ]:
dc.execute('CREATE OR REPLACE TABLE Countries AS SELECT * FROM Countries')
dc.execute('CREATE OR REPLACE TABLE City AS SELECT * FROM City')
dc.execute('CREATE OR REPLACE TABLE Customers AS SELECT * FROM customers')
dc.execute('CREATE OR REPLACE TABLE Categories AS SELECT * FROM categories')
dc.execute('CREATE OR REPLACE TABLE Products AS SELECT * FROM products')
dc.execute('CREATE OR REPLACE TABLE Employees AS SELECT * FROM employees')
dc.execute('CREATE OR REPLACE TABLE Sales AS SELECT * FROM sales')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
tables = dc.execute("SHOW TABLES;").fetchdf()
print("Tables in GrocerySales.db:")
for TabName in tables['name']:
    print(f"\n--- Schema for table: {TabName} ---")
    Schema123 = dc.execute(f"PRAGMA table_info('{TabName}');").fetchdf()
    display(Schema123)

Tables in GrocerySales.db:

--- Schema for table: Categories ---


,cid,name,type,notnull,dflt_value,pk
0,0,CategoryID,BIGINT,False,None,False
1,1,CategoryName,VARCHAR,False,None,False



--- Schema for table: City ---


,cid,name,type,notnull,dflt_value,pk
0,0,CityID,BIGINT,False,None,False
1,1,CityName,VARCHAR,False,None,False
2,2,Zipcode,BIGINT,False,None,False
3,3,CountryID,BIGINT,False,None,False



--- Schema for table: Countries ---


...

,cid,name,type,notnull,dflt_value,pk
0,0,EmployeeID,BIGINT,False,None,False
1,1,FirstName,VARCHAR,False,None,False
2,2,MiddleInitial,VARCHAR,False,None,False
3,3,LastName,VARCHAR,False,None,False
4,4,BirthDate,VARCHAR,False,None,False
5,5,Gender,VARCHAR,False,None,False
6,6,CityID,BIGINT,False,None,False
7,7,HireDate,VARCHAR,False,None,False



--- Schema for table: Products ---


,cid,name,type,notnull,dflt_value,pk
0,0,ProductID,BIGINT,False,None,False
1,1,ProductName,VARCHAR,False,None,False
2,2,Price,DOUBLE,False,None,False
3,3,CategoryID,BIGINT,False,None,False
4,4,Class,VARCHAR,False,None,False
5,5,ModifyDate,VARCHAR,False,None,False
6,6,Resistant,VARCHAR,False,None,False
7,7,IsAllergic,VARCHAR,False,None,False
8,8,VitalityDays,DOUBLE,False,None,False



--- Schema for table: Sales ---


,cid,name,type,notnull,dflt_value,pk
0,0,SalesID,BIGINT,False,None,False
1,1,SalesPersonID,BIGINT,False,None,False
2,2,CustomerID,BIGINT,False,None,False
3,3,ProductID,BIGINT,False,None,False
4,4,Quantity,BIGINT,False,None,False
5,5,Discount,DOUBLE,False,None,False
6,6,TotalPrice,DOUBLE,False,None,False
7,7,SalesDate,VARCHAR,False,None,False
8,8,TransactionNumber,VARCHAR,False,None,False


In [ ]:
query = """
SELECT
    s.SalesID,
    s.SalesDate,
    s.Quantity,
    s.TotalPrice,
    p.ProductName,
    p.Price AS ProductPrice,
    c.CategoryName,
    cust.FirstName AS CustomerFirstName,
    cust.LastName AS CustomerLastName,
    city.CityName,
    ctr.CountryName
FROM
    Sales s
LEFT JOIN
    Products p ON s.ProductID = p.ProductID
LEFT JOIN
    Categories c ON p.CategoryID = c.CategoryID
LEFT JOIN
    Customers cust ON s.CustomerID = cust.CustomerID
INNER JOIN
    City city ON cust.CityID = city.CityID
INNER JOIN
    Countries ctr ON city.CountryID = ctr.CountryID
LIMIT 10;
"""

joined_data = dc.execute(query).fetchdf()
display(joined_data)

,SalesID,SalesDate,Quantity,TotalPrice,ProductName,ProductPrice,CategoryName,CustomerFirstName,CustomerLastName,CityName,CountryName
0,1,2018-02-05 07:38:25.430,7,0.0,Vaccum Bag 10x13,44.2337,Confections,Susan,Green,Albuquerque,United States
1,2,2018-02-02 16:03:31.150,7,0.0,Sardines,62.5460,Grain,Telly,Pollard,Phoenix,United States
2,3,2018-05-03 19:31:56.880,24,0.0,Crab - Imitation Flakes,79.0184,Produce,Jon,Rangel,Buffalo,United States
3,4,2018-04-07 14:43:55.420,19,0.0,Smirnoff Green Apple Twist,81.3167,Seafood,Carol,Gilmore,Dallas,United States
4,5,2018-02-12 15:37:03.940,9,0.0,Coffee - Dark Roast,79.9780,Poultry,Terra,Carter,Charlotte,United States
5,6,2018-02-07 10:33:24.990,8,0.0,Ice Cream Bar - Oreo Cone,95.4065,Cereals,Alexis,Austin,Wichita,United States
6,7,2018-03-02 23:09:58.750,12,0.0,Muffin - Carrot Individual Wrap,21.8806,Dairy,Jeff,Curry,Lincoln,United States
7,8,2018-01-17 13:41:38.460,4,0.0,Bread - Italian Roll With Herbs,57.7090,Meat,Whitney,Sullivan,Baltimore,United States
8,9,2018-04-27 06:19:58.570,23,0.0,Macaroons - Two Bite Choc,18.2891,Dairy,Tracie,Pope,Portland,United States
9,10,2018-03-26 22:12:08.530,17,0.0,Cheese - Parmesan Grated,17.1914,Confections,Cesar,Cain,Memphis,United States


In [ ]:
JoinQuery = '''
CREATE OR REPLACE TABLE
    Master AS (
        SELECT
            SL.SalesID,
            CAST(SL.SalesDate AS DATE) AS SalesDate,
            CAST(SL.SalesDate AS TIME) AS SalesHours,
            SL.CustomerID,
            CU.FirstName,
            PR.ProductName,
            PR.Price AS ProductPrice,
            SL.Quantity,
            SL.Discount,
            CASE
                WHEN (SL.Quantity IS NULL OR SL.Quantity = 0) AND (PR.Price IS NOT NULL OR PR.Price != 0)
                    THEN CAST(PR.Price * (1 - SL.Discount) AS DECIMAL(18,3))
                WHEN (SL.Quantity IS NOT NULL OR SL.Quantity != 0) AND (PR.Price IS NOT NULL OR PR.Price != 0)
                    THEN CAST(PR.Price * SL.Quantity * (1 - SL.Discount) AS DECIMAL(18,3))
                ELSE 0.0
                END AS TotalPrice,
            PR.CategoryID,
            PR.Class,
            PR.Resistant,
            PR.IsAllergic,
            PR.VitalityDays,
            CAST(PR.ModifyDate AS DATE) AS Product_ModifyDate,
            CT.CityName,
            CY.CountryName,
            EP.EmployeeID,
            EP.FirstName AS EmployeeFirstName,
            date_diff('year', CAST(EP.BirthDate AS DATE), CURRENT_DATE) AS EmployeeAge,
            EP.Gender AS EmployeeGender,
            date_diff('year', CAST(EP.HireDate AS DATE), CURRENT_DATE) AS YearsWorking,
            EP.Employee_City
        FROM
            Sales AS SL
        LEFT JOIN
            Customers AS CU ON SL.CustomerID = CU.CustomerID
        LEFT JOIN
            Products AS PR ON SL.ProductID = PR.ProductID
        LEFT JOIN
            City CT ON CU.CityID = CT.CityID
        LEFT JOIN
            Countries CY ON CT.CountryID = CY.CountryID
        LEFT JOIN
            (SELECT A1.*, A2.CityName AS Employee_City
            FROM Employees AS A1 LEFT JOIN City AS A2 ON A1.CityID = A2.CityID)
            AS EP ON SL.SalesPersonID = EP.EmployeeID
        ORDER BY SL.SalesDate ASC
)
'''

dc.execute(JoinQuery)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
Check = dc.execute('SELECT COUNT(*) FROM Master;').fetchdf()
display(Check)

,count_star()
0,6758125


In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

JustSampling = """
SELECT *
FROM Master
TABLESAMPLE 7 ROWS;
"""

Check2 = dc.execute(JustSampling).fetchdf()
display(Check2)

,SalesID,SalesDate,SalesHours,CustomerID,FirstName,ProductName,ProductPrice,Quantity,Discount,TotalPrice,CategoryID,Class,Resistant,IsAllergic,VitalityDays,Product_ModifyDate,CityName,CountryName,EmployeeID,EmployeeFirstName,EmployeeAge,EmployeeGender,YearsWorking,Employee_City
0,4733820,2018-01-01,00:40:29.110000,80312,Ruby,Beef - Ground Medium,79.5324,21,0.0,1670.180,4,Medium,Weak,True,0.0,2018-03-23,Cleveland,United States,7,Chadwick,55,M,9,Lubbock
1,2206020,2018-01-03,08:53:26.170000,15841,Willie,Beef - Striploin Aa,2.3890,5,0.0,11.945,2,Medium,Weak,Unknown,0.0,2017-06-09,Tulsa,United States,18,Warren,61,M,15,Columbus
2,5158196,2018-01-03,11:44:57.230000,651,Tammi,Ketchup - Tomato,68.6380,1,0.1,61.774,11,High,Weak,Unknown,0.0,2018-03-27,Des Moines,United States,12,Lindsay,74,F,14,Columbus
3,5714637,2018-01-03,18:01:28.210000,41119,Wendi,Rice - Jasmine Sented,27.8955,11,0.0,306.851,6,Medium,Weak,Unknown,48.0,2017-05-09,Omaha,United States,8,Julie,69,M,11,Little Rock
4,5868838,2018-01-03,19:10:56.800000,61247,Latisha,Veal - Eye Of Round,67.4367,16,0.0,1078.987,7,High,Unknown,True,71.0,2017-04-10,San Diego,United States,23,Janet,46,F,15,Riverside
5,2413452,2018-01-03,20:10:27.990000,76682,Paula,Pasta - Cheese / Spinach Bauletti,55.3683,20,0.0,1107.366,10,Low,Durable,True,54.0,2017-11-29,Seattle,United States,3,Pablo,62,M,13,Rochester
6,3569325,2018-01-03,20:51:04.410000,39349,Lamar,"Oregano - Dry, Rubbed",68.1326,10,0.0,681.326,11,High,Durable,Unknown,16.0,2018-04-04,Long Beach,United States,11,Sonya,49,F,9,Tacoma


In [ ]:
dc.execute("COPY Master TO 'MasterData.parquet' (FORMAT PARQUET, COMPRESSION 'ZSTD', COMPRESSION_LEVEL 22);")
print("MasterData.parquet has been created with ZSTD compression (level 22).")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

MasterData.parquet has been created with ZSTD compression (level 22).


In [ ]:
DirNow = os.getcwd()
print(f"Files and their sizes in {DirNow}:")

def GetFileSize(dpath : str) -> None:
    for item in os.listdir(dpath):
        item_path = os.path.join(dpath, item)
        if os.path.isfile(item_path):
            size_bytes = os.path.getsize(item_path)
            size = size_bytes / (1024 ** 2)
            print(f"  {item}: {size:.2f} MB")

GetFileSize(DirNow)

Files and their sizes in /content/GrocerySales:
  categories.csv: 0.00 MB
  cities.csv: 0.00 MB
  countries.csv: 0.00 MB
  MasterData.parquet: 145.61 MB
  sales.csv: 493.07 MB
  employees.csv: 0.00 MB
  GrocerySales.db: 397.51 MB
  products.csv: 0.04 MB
  customers.csv: 4.23 MB


In [ ]:
EmpMost = """
SELECT
    EmployeeAge,
    EmployeeFirstName,
    EmployeeGender,
    Employee_City,
    COUNT(*) AS TotalSales
FROM
    Master
GROUP BY
    EmployeeAge,
    EmployeeFirstName,
    EmployeeGender,
    Employee_City
ORDER BY
    EmployeeAge DESC;
"""

Check3 = dc.execute(EmpMost).fetchdf()
display(Check3)

,EmployeeAge,EmployeeFirstName,EmployeeGender,Employee_City,TotalSales
0,74,Lindsay,F,Columbus,293164
1,74,Chadwick,M,Tucson,293685
2,73,Tonia,F,Las Vegas,293224
3,69,Julie,M,Little Rock,294449
4,69,Daphne,F,Lubbock,294180
5,64,Wendi,M,Jackson,294035
6,62,Pablo,M,Rochester,293175
7,62,Desiree,F,Anaheim,293711
8,62,Katina,M,Anchorage,293530
9,62,Jean,M,Atlanta,293888


In [ ]:
EmpMost = """
SELECT
    EmployeeID,
    EmployeeFirstName,
    EmployeeGender,
    COUNT(*) AS Number_Sales,
    SUM(Quantity) AS Total_Quantity,
    MEAN(ProductPrice) AS Average_ProductPrice,
    MEAN(TotalPrice) AS Average_TotalPrice,
    SUM(TotalPrice) AS All_Product_SalesPrice,
    MEAN(Discount) * 100 AS Average_Discount_Percentage
FROM
    Master
GROUP BY
    EmployeeID,
    EmployeeFirstName,
    EmployeeGender
ORDER BY
    EmployeeGender DESC;
"""

Check3 = dc.execute(EmpMost).fetchdf()
display(Check3)

,EmployeeID,EmployeeFirstName,EmployeeGender,Number_Sales,Total_Quantity,Average_TotalPrice,All_Product_SalesPrice,Average_Discount_Percentage
0,10,Jean,M,293888,3820273.0,637.196510,1.872644e+08,3.017136
1,8,Julie,M,294449,3832712.0,641.410205,1.888626e+08,2.984354
2,16,Chadwick,M,293685,3817830.0,639.391975,1.877798e+08,2.980779
3,7,Chadwick,M,293967,3819981.0,642.232082,1.887950e+08,3.003807
4,4,Darnell,M,294744,3831150.0,641.695760,1.891360e+08,3.003182
5,21,Devon,M,294983,3841871.0,644.249845,1.900428e+08,2.979562
6,6,Holly,M,293973,3824169.0,642.184299,1.887848e+08,3.006467
7,20,Shelby,M,293562,3818823.0,645.223663,1.894131e+08,2.989965
8,19,Bernard,M,293875,3822931.0,641.286570,1.884581e+08,3.006652
9,18,Warren,M,294419,3831614.0,639.287079,1.882183e+08,2.988598


In [ ]:
from google.colab import drive
from shutil import copy2

drive.mount('/content/drive')
source_path = './MasterData.parquet'
destination_folder = '/content/drive/My Drive/Colab Notebooks'

os.makedirs(destination_folder, exist_ok=True)
copy2(source_path, destination_folder)

print(f"'{source_path}' successfully exported to '{destination_folder}'")

Mounted at /content/drive
'./MasterData.parquet' successfully exported to '/content/drive/My Drive/Colab Notebooks'
